# Step 1 -- Extract PDFs files from RTRS web Site


In [152]:
import requests, os, re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By 
from time import sleep

options = webdriver.ChromeOptions()

options.add_argument('user-data-dir=Perfil')
#options.add_argument('--headless')

navegador = webdriver.Chrome(
            'chromedriver',
            options=options)

navegador.get('https://responsiblesoy.org/public-audit-reports?lang=en')

navegador.find_element(By.CLASS_NAME, 'zyxaudit-btn_buscar').click()

sleep(2)

#integrate bs4 and Selenium (when Vegeta and Goku merge!)

web_html = BeautifulSoup(navegador.page_source, 'html.parser')

navegador.quit()

audit_general_data = web_html.find('div', attrs={'class': 
                                              'zyxaudit-buscador_results vc_col-xs-12 col-xs-12 zyxaudit-results-active'})

names = []
country = []
year = []
links = []

labels = ['vc_col-md-6 vc_col-xs-12 zyxaudit-nombre_empresa',
         'vc_col-md-2 vc_col-xs-12 zyxaudit-pais', 
         'vc_col-md-2 vc_col-xs-12 zyxaudit-anio', ]

for clas in labels:
    for audit in audit_general_data:
        temp = audit.find('div', attrs = {'class':clas})
        if clas.endswith('nombre_empresa'):
            names.append(temp.text)
        elif clas.endswith('pais'):
            country.append(temp.text)
        else:
            year.append(temp.text)
            
#links to download audit reports             
a_tags = audit_general_data.find_all('a', href = True)

for tag in a_tags:
    links.append(tag['href'])        

#create a data frame obj

certified_producers_rtrs = pd.DataFrame(list(zip(names[1:325],
                                                country[1:325],
                                                year[1:325],
                                                links)), 
                                        columns = ['Names', 'Country', 'Year', 'Links'])

certified_producers_rtrs['Year'] = pd.to_numeric(certified_producers_rtrs['Year'])

certified_producers_rtrs.to_csv('certified_producers_rtrs.csv')


In [161]:
folder_location = r'/home/cleyton/Dropbox/naea_pos_grad/qualificacao_dissertacao/rtrs_py/audit_reports_files'
if not os.path.exists(folder_location): os.mkdir(folder_location) #If doesn't exist a folder_location, python create a one
       
x = 0

url = 'https://responsiblesoy.org/wp-content/uploads/2022/12/'

for link in links:
    REname = re.sub('[^0-9a-zA-Z\[\]]+', ' ', str(link))
    
    filename = os.path.join(folder_location, REname + ".pdf")
    
    print(filename)
    
    x += 1
    print('Download file: ', x)
    
    with open(filename, 'wb') as f:
        f.write(requests.get(urljoin(url, link).content))

/home/cleyton/Dropbox/naea_pos_grad/qualificacao_dissertacao/rtrs_py/audit_reports_files/https responsiblesoy org wp content uploads 2022 12 Castrolanda II Relatorio de Resumo Publico 2022 1 pdf.pdf
Download file:  1


NameError: name 'urljoin' is not defined

In [159]:
links

['https://responsiblesoy.org/wp-content/uploads/2022/12/Castrolanda-II-Relatorio-de-Resumo-Publico-2022-1.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/Castrolanda-Relatorio-Resumo-Publico-RTRS-2022-2.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/2022-Public-summary-Cusillos-3-surveillance-1.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/2022-Cosufi-Public-Summary-2o-surveillance.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/2022-Resumo-Publico-Girassol-Agricola.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/2022-875207-Public-Summary-2-surveillance-Fazenda-modelo.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/LA-ORIENTAL-Informe_Publico_de_Certificacion-3o-SVS-2022.2APROBADO1.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/SyDLafuente-SA-SV2022-Informe_Publico_de_Certificacion_RTRS.traducido-APROBADO.MAIZnb2.pdf',
 'https://responsiblesoy.org/wp-content/uploads/2022/12/2022-Harit